# L4a: Graph and Tree Representations

A graph stores objects, called vertices, and links between pairs of objects, called edges. Roads between cities, one chemical species changing into another, and calls between functions can all be drawn as graphs. A tree is an undirected graph that is connected and has no cycles. It has exactly enough edges to connect its vertices: removing any edge disconnects it. 

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
>
> * __Describe a graph and its measures:__ Define vertices, edges, walks, paths, cycles, and connectivity, and distinguish weak from strong connectivity in a directed graph. Compute vertex degrees and graph density from the vertex and edge counts.
> * __Recognize complete graphs, bipartite graphs, and trees:__ Identify a complete graph by an edge between every pair of vertices, a bipartite graph by a two-coloring, and a tree by connectivity with no cycles. Use edge counts, colors, and paths to tell the three families apart.
> * __Choose a storage representation:__ Read a graph from an edge list into an adjacency list and an adjacency matrix. Choose between the two from the density of the graph and from whether the algorithm mostly checks for a single edge or mostly loops over the neighbors of a vertex.

In this lecture, we define graph types, compute measures that describe the different properties of graphs, and compare three ways to store a graph in memory. Let's get started!

___

## Setup and Data

Run the next cell to load [`Include.jl`](Include.jl) and the code used in this lecture.

> The [`include(...)` function](https://docs.julialang.org/en/v1/base/base/#include) runs the Julia code in [`Include.jl`](Include.jl). That file loads the course environment and graph functions. It also sets paths from the lecture folder. See the [Julia programming language documentation](https://docs.julialang.org/en/v1/) for Julia functions and types.

Run the setup cell:

In [ ]:
# Setup -
include(joinpath(@__DIR__, "Include.jl")); # load the course environment and graph functions

The lecture uses [the `Test` standard library](https://docs.julialang.org/en/v1/stdlib/Test/) for the checks and graph functions from [`GraphRepresentation.jl`](../../../code/src/GraphRepresentation.jl). The worked example reads the sample edge list in [`data/SimpleGraph.txt`](data/SimpleGraph.txt), which is documented in [`data/README.md`](data/README.md).

___

## Simple Graphs

A simple graph $\mathcal{G} = (\mathcal{V},\mathcal{E})$ consists of a vertex set $\mathcal{V}$ and an edge set $\mathcal{E}$. In a simple undirected graph, an edge is an unordered pair $\{u,v\}$ of distinct vertices. In a simple directed graph, an edge is an ordered pair $(u,v)$ with $u\neq v$; the reverse edge $(v,u)$ is a different edge. The word *simple* rules out self-loops and repeated parallel edges. Either kind of edge may carry a weight such as distance, capacity, or cost.

The direction tells us how an edge can be followed:

* In a __directed__ graph, $(u,v)$ points from $u$ to $v$. It does not mean that the reverse edge $(v,u)$ exists. In a social network, person A may follow person B without B following A.
* In an __undirected__ graph, $\{u,v\}$ can be followed in either direction.

A __walk__ is a sequence $v_0,v_1,\ldots,v_k$ in which every consecutive pair is joined by an edge; a directed walk must follow each edge from source to target. A __path__ is a walk with no repeated vertices. A __cycle__ is a closed walk that repeats no edge and repeats no vertex except that $v_0=v_k$. An undirected graph is __connected__ when a path joins every pair of vertices. A directed graph is __weakly connected__ when it becomes connected after edge directions are ignored, and __strongly connected__ when every ordered pair $(u,v)$ is joined by a directed path from $u$ to $v$.

<div>
    <center>
        <img src="figs/Fig-General-Graph-Schematic.svg" width="980" alt="Left: an undirected graph with six vertices and seven weighted edges, with the degree of vertex 4 marked. Right: the same vertices and edges drawn as a directed acyclic graph, with the in-degree of vertex 2 and the out-degree of vertex 4 marked."/>
    </center>
</div>

### Graph Measures

The __degree__ $\deg(v_i)$ of a vertex $v_i\in\mathcal{V}$ counts the edges that touch it. In a directed graph, we split this count by direction:

* __In-degree__ $\deg^{\text{in}}(v_{i})$: the number of edges that point into $v_{i}$.
* __Out-degree__ $\deg^{\text{out}}(v_{i})$: the number of edges that point out of $v_{i}$.

The total degree of a vertex in a directed graph is $\deg(v_{i}) = \deg^{\text{in}}(v_{i}) + \deg^{\text{out}}(v_{i})$. In the figure, vertex 4 has degree 2 in the undirected graph. In the directed graph, vertex 2 has in-degree 2 and vertex 4 has out-degree 1.

> __Degree, edge count, and density:__
>
> Let $n=|\mathcal{V}|$ be the number of vertices and $m=|\mathcal{E}|$ be the number of edges. In an undirected graph, the sum of the vertex degrees counts both ends of every edge. In a directed graph, the sum of the in-degrees and the sum of the out-degrees each count every edge once. These facts give the __handshaking identities__:
> $$
> \begin{align*}
> \sum_{v_i\in\mathcal{V}}\deg(v_i) &= 2m && \text{(undirected)},\\
> \sum_{v_i\in\mathcal{V}}\deg^{\mathrm{in}}(v_i) &= \sum_{v_i\in\mathcal{V}}\deg^{\mathrm{out}}(v_i)=m && \text{(directed)}.
> \end{align*}
> $$
>
> For an undirected graph with $n\geq1$ vertices, the __minimum degree__ $\delta(\mathcal{G})$ and __maximum degree__ $\Delta(\mathcal{G})$ bound the __average degree__ $\bar d(\mathcal{G})$ as follows:
> $$
> \begin{align*}
> \bar d(\mathcal{G}) &= \frac{1}{n}\sum_{v_i\in\mathcal{V}}\deg(v_i)=\frac{2m}{n},\\
> \delta(\mathcal{G}) &\leq \bar d(\mathcal{G}) \leq \Delta(\mathcal{G}).
> \end{align*}
> $$
> An undirected graph is __regular__ of degree $r$ when every vertex has degree $r$. For a simple graph with $n\geq2$ vertices, the maximum edge count and the __density__ $\rho(\mathcal{G})$ are given by:
> $$
> \begin{align*}
> |\mathcal{E}|_{\max} &= \begin{cases}n(n-1)/2 & \text{undirected},\\ n(n-1) & \text{directed},\end{cases}\\
> \rho(\mathcal{G}) &= \frac{|\mathcal{E}|}{|\mathcal{E}|_{\max}}.
> \end{align*}
> $$
> A graph with $\rho$ close to 1 is __dense__. A graph with $\rho$ close to 0 is __sparse__. When $n<2$, no loop-free edges are possible, so this lecture sets $\rho(\mathcal{G})=0$, matching [the `directed_density(...)` implementation](../../../code/src/GraphRepresentation.jl).

Degree and density do not show how edges are arranged. For $n\geq4$, a path graph and a star graph can both have $n$ vertices and $n-1$ edges. The path has maximum degree 2 and diameter $n-1$; the star has maximum degree $n-1$ and diameter 2. For an undirected graph, the next four measures answer questions about distance and which vertices may be grouped together:

* __Diameter__ $\text{diam}(\mathcal{G})$: for a connected graph, the largest shortest-path distance between two vertices, measured in edges.
* __Clique number__ $\omega(\mathcal{G})$: the size of the largest set of vertices in which every pair is joined, such as the largest group of people who all know each other.
* __Chromatic number__ $\chi(\mathcal{G})$: the fewest colors needed so that no edge joins two vertices of the same color, such as the fewest time slots that schedule a set of classes with no student in two classes at once.
* __Independence number__ $\alpha(\mathcal{G})$: the size of the largest set of vertices with no edge among them, such as the most activities that can run at the same time with no conflict.

Vertex and edge counts alone do not determine these four measures. Computing them requires edge lookups and neighbor lists.

___

## How are Graphs Stored?

An algorithm needs to look up edges and find the neighbors of a vertex. We compare three ways to store the graph: an edge list, an adjacency matrix, and an adjacency list.

An __edge list__ stores one record per edge, containing its endpoints and, when present, its weight. Text files often use this format, including the worked example below. Without an index, finding one edge or all neighbors of a vertex requires scanning the records.

An __adjacency matrix__ $\mathbf{A}$ for a graph with $|\mathcal{V}|$ vertices is a $|\mathcal{V}|\times|\mathcal{V}|$ matrix. The entry $a_{ij}$ in row $i$ and column $j$ describes the edge from $v_{i}$ to $v_{j}$:

* __Unweighted__: $a_{ij}=1$ if the edge exists and $a_{ij}=0$ if it does not.
* __Weighted__: $a_{ij}=w_{ij}$, the weight of the edge, if the edge exists, and $a_{ij}=0$ if it does not.

For an undirected graph the matrix is symmetric, $a_{ij} = a_{ji}$; for a directed graph it need not be. In the weighted form, a stored zero can mean either a missing edge or an edge of weight zero, so zero-weight edges require a separate marker for missing edges.

An __adjacency list__ is a dictionary with one entry per vertex. The entry for $v_{i}$ holds the set $\mathcal{C}_{i}$ of vertices that $v_{i}$ is joined to. In a directed graph these are the vertices its edges point to, the out-neighbors. When weights are needed, they are stored beside each neighbor.

> __What does it cost to store and use a graph?__
>
> Let $n=|\mathcal{V}|$ be the number of vertices and $m=|\mathcal{E}|$ be the number of edges. All three forms store the same graph, but the number and layout of memory entries differ.
>
> * An __edge list__ stores one `(source, target, weight)` record for each edge, so it uses $O(m)$ records. Finding one edge or collecting every edge leaving $v_i$ can require scanning all $m$ records.
> * An __adjacency matrix__ stores one entry $a_{ij}$ for every ordered pair of vertices, so it uses exactly $n^2$ entries whether or not most pairs are connected. The entry for $(v_i,v_j)$ can be read in $O(1)$ time, while finding every neighbor of $v_i$ requires scanning its $n$ matrix entries.
> * An __adjacency list__ stores one neighbor collection for each vertex and one target entry for each directed edge, so it uses $n+m$ stored items. Accessing a vertex's vector in [the `Dict{Int64,Vector{Int64}}` type](https://docs.julialang.org/en/v1/base/collections/#Dictionaries) used here takes expected $O(1)$ time. Iterating over every out-neighbor of $v_i$, or testing for one target by a linear scan, takes $O(\deg^{\mathrm{out}}(v_i))$ time. A weighted list would store a `(target, weight)` pair instead of only the target.
>
> Ignoring container overhead and the fixed number of fields in an edge record, storage grows as follows:
> $$
> S_{\mathrm{edge\ list}}=O(m),\qquad S_{\mathrm{matrix}}=O(n^2),\qquad S_{\mathrm{adjacency\ list}}=O(n+m).
> $$
> For an undirected adjacency list, each edge appears in both endpoint lists, giving $n+2m$ stored items. Storage still grows as $O(n+m)$.

Choose a representation based on what the algorithm needs to do. An adjacency matrix stores all $n^2$ vertex pairs and answers one edge lookup in $O(1)$ time. An adjacency list stores no entries for missing edges and scans only the out-neighbors of a vertex. Use a matrix for a dense graph with repeated edge lookups. Use a list to traverse a sparse graph. [The L4b traversal lab](../L4b/CHEME-5800-L4b-Lab-BreadthFirstAndDepthFirstSearch-Fall-2026.ipynb) uses an adjacency list because breadth-first and depth-first search read the out-neighbors of every visited vertex.

### Worked Example: One Graph in Three Forms

The seven records in [`data/SimpleGraph.txt`](data/SimpleGraph.txt) store a directed graph with six vertices and one weight per edge. [The L4b traversal lab](../L4b/CHEME-5800-L4b-Lab-BreadthFirstAndDepthFirstSearch-Fall-2026.ipynb) uses a copy of the same data. [The `read_weighted_edges(...)` function](../../../code/src/GraphRepresentation.jl) loads the edge records, [the `adjacency_list(...)` function](../../../code/src/GraphRepresentation.jl) builds the outgoing-neighbor lists, and [the `adjacency_matrix(...)` function](../../../code/src/GraphRepresentation.jl) builds the weighted matrix.

The adjacency list keeps the out-neighbors and drops the weights; the matrix keeps the weights. Both functions get the vertex set from the edge endpoints, so an isolated vertex would need a separate vertex record or an input list of vertices. The next cell stores the edge records in `edge_records::Vector{<:NamedTuple}`, the adjacency list in `adjacency::Dict{Int64, Vector{Int64}}`, and the matrix with its row and column order in `matrix_representation::NamedTuple`.

In [ ]:
# Build three representations of the same directed, weighted graph -
# The let block keeps temporary names inside it and returns the three notebook values.
edge_records, adjacency, matrix_representation = let
    # Locate and read the edge list -
    edge_path = joinpath(CHEME5800_L4A_DATA, "SimpleGraph.txt") # start from the L4a data folder, not pwd()
    records = read_weighted_edges(edge_path)                    # Vector of (source, target, weight) NamedTuples

    # Convert the edge records into two graph data structures -
    list = adjacency_list(records)                               # Dict: vertex id => sorted outgoing-neighbor ids
    matrix = adjacency_matrix(records)                           # NamedTuple: weighted matrix plus row/column vertex ids

    # Return values from the local scope -
    records, list, matrix                                       # return the three representations from let
end; # hide the full output; display the three stored values in the next cell

The two outputs store the same directed edges in different forms. The value `vertex_order[i]` gives the vertex for row and column `i`, so the vertex IDs do not need to equal `1:n`.

In [ ]:
# Display the out-neighbor list beside the equivalent weighted matrix -
(
    adjacency = adjacency,                            # each key is a vertex; each value is its sorted out-neighbor vector
    vertex_order = matrix_representation.vertex_ids, # position i in this vector identifies matrix row and column i
    matrix = matrix_representation.matrix,            # entry (i,j) is the weight from vertex_order[i] to vertex_order[j]
)

Vertex 1 points to vertices 2 and 3, so `adjacency[1]` contains `[2, 3]` and row 1 of the matrix contains weights 10 and 100 in columns 2 and 3. Vertex 6 has an empty neighbor vector because no edge leaves it.

[The `representation_report(...)` function](../../../code/src/GraphRepresentation.jl) returns the vertex and edge counts, the directed density $|\mathcal{E}|/(|\mathcal{V}|(|\mathcal{V}|-1))$, the $|\mathcal{V}|^{2}$ matrix entries, and the $|\mathcal{V}|+|\mathcal{E}|$ adjacency-list items. The next cell also computes storage for 100,000 vertices with ten outgoing edges per vertex, using 8 bytes per matrix entry or list item and excluding container overhead. The six-vertex report is stored in `representation::NamedTuple`.

In [ ]:
# Compare storage for the six-vertex graph and a larger sparse graph -
# The let block returns only the six-vertex report; the large-graph values stay inside the block.
representation = let
    # Measure the graph read from SimpleGraph.txt -
    report = representation_report(edge_records)                    # graph measures and storage-entry counts

    # Set the size of a larger sparse graph for the storage comparison -
    n, k = 100_000, 10                                              # n vertices, k outgoing edges each, and m = n⋅k
    bytes_per_entry = sizeof(Float64)                               # Float64 weights and Int64 ids each use 8 bytes

    # Estimate storage without counting container overhead -
    matrix_gigabytes = n^2 * bytes_per_entry / 1e9                  # n² weighted entries, converted to decimal gigabytes
    adjacency_list_megabytes = (n + n * k) * bytes_per_entry / 1e6  # n keys plus n⋅k targets, in decimal MB

    # Display the large-graph comparison and return the six-vertex report -
    println("Large sparse graph: matrix ≈ $(matrix_gigabytes) GB, adjacency list ≈ $(adjacency_list_megabytes) MB")
    report                                                         # return the six-vertex NamedTuple from let
end

For the six-vertex graph, the matrix reserves 36 entries. The directed adjacency list contains 6 vertex keys and 7 neighbor slots. For 100,000 vertices with mean out-degree 10, the matrix contains $10^{10}$ entries and occupies 80 decimal gigabytes at 8 bytes per entry. The list contains $1.1\times10^6$ items and occupies 8.8 decimal megabytes before container overhead. Matrix storage grows as $n^2$; list storage grows as $n+m$. The following tests check the six-vertex counts, density, adjacency list, and one matrix weight against the input file.

In [ ]:
# Check that every representation stores the expected graph -
@testset "graph representations" begin
    # Check the structure encoded by the input and both representations -
    @test representation.vertices == 6                       # the edge endpoints use the six vertex ids 1,...,6
    @test representation.edges == 7                          # the file contains seven directed source-to-target records
    @test representation.density ≈ 7 / 30                    # 7 observed edges divided by 6(6-1) possible loop-free edges
    @test adjacency[1] == [2, 3]                             # vertex 1 points outward to vertices 2 and 3, in sorted order
    @test matrix_representation.matrix[1, 2] == 10.0         # entry (1,2) stores the weight of edge 1 → 2

    # Check the exact storage counts used in the lecture comparison -
    @test representation.matrix_entries == 36                # a 6×6 matrix reserves one entry for every ordered pair
    @test representation.adjacency_list_entries == 13        # six dictionary keys plus one target entry per edge
end

___

## Graph Families

### Complete Graphs

A complete graph $K_{n}$ is a simple undirected graph on $n$ vertices with an edge between every pair of distinct vertices. Once $n$ is known, the graph is fixed except for the vertex names.

> __Counts and measures for $K_n$:__
>
> Every vertex touches the other $n-1$ vertices, so $K_n$ is regular of degree $n-1$. Every possible edge is present, and every pair of vertices is one edge apart. For $n\geq 2$, the edge count, density, and diameter are given by:
> $$
> \begin{align*}
> |\mathcal{E}| &= \binom{n}{2}=\frac{n(n-1)}{2}, & \deg(v_i) &= n-1,\\
> \rho(K_n) &= 1, & \operatorname{diam}(K_n) &= 1.
> \end{align*}
> $$
> The full vertex set is a clique, adjacent vertices require different colors, and an independent set can contain only one vertex. These facts give:
> $$
> \omega(K_n)=\chi(K_n)=n,\qquad \alpha(K_n)=1.
> $$

A round-robin tournament has the edge pattern of $K_n$: every team plays every other team. For an algorithm whose work grows with $|\mathcal{E}|$, $K_n$ has the largest edge count possible for an undirected simple graph on $n$ vertices. The traveling-salesman problem uses a weighted complete graph when every city pair has a travel cost and the goal is to find the tour with the least cost.

### Bipartite Graphs

A graph $\mathcal{G}=(\mathcal{V},\mathcal{E})$ is bipartite if its vertices can be split into two groups, $\mathcal{V}_{1}$ and $\mathcal{V}_{2}$, with no vertex in both groups. Every edge has one endpoint in each group, so no edge joins two vertices in the same group.

<div>
    <center>
        <img src="figs/Fig-Bipartite-Graph-Schematic.png" width="280" alt="A bipartite graph drawn with four vertices in a left column and eight in a right column; every edge crosses from the left column to the right column."/>
    </center>
</div>

The figure shows $K_{4,8}$. It has four vertices in $\mathcal{V}_{1}$, eight vertices in $\mathcal{V}_{2}$, and all $4\times8=32$ edges between the two groups. In general, let $m=|\mathcal{V}_{1}|\geq1$ and $n=|\mathcal{V}_{2}|\geq1$. The __complete bipartite graph__ $K_{m,n}$ has all $mn$ edges between the groups.

A __matching__ is a set of edges that do not share vertices. In an assignment graph, a matching that covers $\mathcal{V}_{1}$ gives every worker or student on the left a different option on the right.

> __Three tests for a bipartite graph:__
>
> For an undirected graph $\mathcal{G}$, the following are equivalent:
>
> 1. $\mathcal{G}$ is bipartite.
> 2. $\mathcal{G}$ can be colored with at most two colors, so $\chi(\mathcal{G}) \leq 2$.
> 3. $\mathcal{G}$ has no cycle of odd length.
>
> Coloring the two groups gives $1\Rightarrow2$. A two-coloring must alternate colors around a cycle, so it cannot close an odd cycle; this gives $2\Rightarrow3$. For $3\Rightarrow1$, start a breadth-first search in each connected component and color vertices by even or odd distance from the start. An edge between two vertices with the same distance parity would create an odd cycle.
>
> In $K_{m,n}$, each vertex of $\mathcal{V}_{1}$ has degree $n$, and each vertex of $\mathcal{V}_{2}$ has degree $m$. The edge count and degrees are:
> $$
> |\mathcal{E}(K_{m,n})|=mn,\qquad \deg(v)=\begin{cases}n & v\in\mathcal{V}_{1},\\ m & v\in\mathcal{V}_{2}\end{cases}.
> $$
> For $m,n\geq1$, the graph $K_{m,n}$ is regular exactly when $m=n$.
>
> __Hall's theorem__ tests whether a matching covers every vertex of $\mathcal{V}_{1}$. For a set $S\subseteq\mathcal{V}_{1}$, let $N(S)\subseteq\mathcal{V}_{2}$ be the set of all neighbors of vertices in $S$. A matching that covers $\mathcal{V}_{1}$ exists if and only if:
> $$
> |N(S)|\geq |S|\qquad\text{for every }S\subseteq\mathcal{V}_{1}.
> $$
> In words, every set of vertices on the left must have at least as many neighbors on the right. A matching is __perfect__ when it covers both groups; this requires $|\mathcal{V}_{1}|=|\mathcal{V}_{2}|$.
>
> List the vertices of $\mathcal{V}_{1}$ before the vertices of $\mathcal{V}_{2}$. The adjacency matrix then has the block form:
> $$
> \mathbf{A}=\begin{bmatrix}\mathbf{0} & \mathbf{B}\\ \mathbf{B}^{\top} & \mathbf{0}\end{bmatrix},
> $$
> where the $m\times n$ __biadjacency matrix__ $\mathbf{B}$ stores the edges between the two groups. The two diagonal blocks are zero because no edge stays within a group.

The coloring test can be carried out with a graph traversal:

1. Mark every vertex uncolored.
2. Choose an uncolored vertex, give it color 1, and run breadth-first or depth-first search through its connected component.
3. Give each uncolored neighbor the opposite color of the current vertex. If an edge joins two vertices with the same color, stop: the graph is not bipartite.
4. Repeat from another uncolored vertex until every connected component has been checked. If no conflict occurs, the two color classes are $\mathcal{V}_{1}$ and $\mathcal{V}_{2}$.

With an undirected adjacency list, this test visits each vertex once and reads each edge from both endpoint lists. Its running time is $O(|\mathcal{V}|+|\mathcal{E}|)$. [The L4b lab](../L4b/CHEME-5800-L4b-Lab-BreadthFirstAndDepthFirstSearch-Fall-2026.ipynb) develops the breadth-first and depth-first searches used by this test.

Bipartite graphs model links between two types of objects: workers and jobs, students and courses, users and items, regulatory proteins and the genes they control, or species and habitats. An adjacency list stores these graphs in the same form as other graphs. In a matrix, storing $\mathbf{B}$ is enough because it determines $\mathbf{B}^{\top}$ and the two zero blocks.

### Trees

A tree $\mathcal{T}=(\mathcal{V},\mathcal{E})$ is a connected undirected graph with no cycles. Connectivity gives at least one path between each pair of vertices. If two such paths existed, their union would contain a cycle, so the path is unique. Removing any edge disconnects the tree. Adding an edge between two vertices that are not already adjacent creates one cycle.

> __Six tests for a tree:__
>
> For an undirected graph $\mathcal{G}$ with $n\geq1$ vertices, the following statements are equivalent:
>
> 1. $\mathcal{G}$ is a tree.
> 2. $\mathcal{G}$ is connected and has exactly $n-1$ edges.
> 3. $\mathcal{G}$ has no cycle and has exactly $n-1$ edges.
> 4. $\mathcal{G}$ is connected, and removing any edge disconnects it.
> 5. $\mathcal{G}$ has no cycle, and adding any missing edge creates exactly one cycle.
> 6. Any two vertices of $\mathcal{G}$ are joined by exactly one path.
>
> Any one of these tests also shows that a tree has $n-1$ edges. For $n\geq2$, its edge count and density are:
> $$
> |\mathcal{E}|=n-1,\qquad \rho(\mathcal{G})=\frac{n-1}{n(n-1)/2}=\frac{2}{n}.
> $$
> The density is $2/n$, so it tends to zero as $n$ grows even though the tree remains connected.

Choose one vertex as the __root__. Every other vertex has one __parent__: the next vertex on its path to the root. A vertex may have zero or more __children__. A vertex with no children is a __leaf__. The __depth__ of a vertex is the number of edges from the root to that vertex. The __height__ of the rooted tree is its largest vertex depth.

<div>
    <center>
        <img src="figs/Fig-General-Tree-Schematic.svg" width="880" alt="A rooted tree drawn top down: the root at depth 0, branch nodes at depths 1 and 2, and leaves at depths 2 and 3, with the empty set marking the missing children of each leaf."/>
    </center>
</div>

The figure shows a rooted tree of height 3. Rooted trees model file systems, organization charts, and function calls because each non-root vertex has one parent.

Every connected undirected graph $\mathcal{G}$ contains a __spanning tree__: a subgraph $\mathcal{T}$ with every vertex of $\mathcal{G}$ and $n-1$ edges that keep the vertices connected. If a graph has no cycles but is not connected, each connected component is a tree. Their union is a __forest__.

Dynamic programming finds a largest independent set of a tree in $O(n)$ time, while the same problem is NP-hard on a general graph.

___

## Looking Ahead: Graph Traversal

A graph traversal starts at one vertex, reads its neighbor list, and repeats this step for each vertex it reaches. It stops when no unvisited reachable vertex remains. We use an adjacency list so the traversal reads only existing neighbors.

In [the L4b lab](../L4b/CHEME-5800-L4b-Lab-BreadthFirstAndDepthFirstSearch-Fall-2026.ipynb), we use the same six-vertex directed graph. Breadth-first search (BFS) uses a queue and visits vertices in layers by their distance from the start. Depth-first search (DFS) follows one branch before it backtracks. The edge weights do not affect either visit order. With an adjacency list, both algorithms run in $O(|\mathcal{V}|+|\mathcal{E}|)$ time.

___

## Summary

A graph states which pairs of vertices are joined. Its storage form sets the cost of looking up one edge and listing the neighbors of one vertex.

> __Key Takeaways:__
>
> * __Count vertices and edges:__ Vertex degree counts the edges that touch a vertex, and graph density is the fraction of possible edges that are present. In an undirected graph, the degree sum is $2|\mathcal{E}|$.
> * __Recognize three graph families:__ A complete graph has every possible edge. An undirected graph is bipartite exactly when it has no odd cycle, or when it can be colored with at most two colors. A tree is connected, has no cycle, has $n-1$ edges, and has one path between each pair of vertices.
> * __Match storage to the operation:__ An adjacency matrix stores $n^2$ entries and checks one edge in $O(1)$ time. A directed adjacency list stores $n+m$ items and lists the out-neighbors of $v_i$ in $O(\deg^{\mathrm{out}}(v_i))$ time.

[The L4b lab](../L4b/CHEME-5800-L4b-Lab-BreadthFirstAndDepthFirstSearch-Fall-2026.ipynb) uses the adjacency list to traverse a graph with breadth-first and depth-first search.

___